# 01 · Data Exploration
Exploratory data analysis of the NHANES 2017-2018 datasets used to train the
Preventive Health Assistant risk models.

**Datasets loaded:**
- DEMO — Demographics (age, gender, education, income)
- BMX  — Body measurements (BMI, waist, weight, height)
- BPX  — Blood pressure (systolic / diastolic)
- GHB  — HbA1c
- GLU  — Fasting blood glucose
- TCHOL / HDL — Cholesterol
- DIQ  — Diabetes self-report
- SMQ  — Smoking
- PAQ  — Physical activity

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='darkgrid', palette='muted')

DATA_DIR = '../data/raw/2017_2018'
print('Notebook ready.')

## 1 · Load raw XPT files

In [ ]:
import pyreadstat

def load_xpt(name):
    path = f'{DATA_DIR}/{name}_2017_2018.XPT'
    df, meta = pyreadstat.read_xport(path)
    return df

demo  = load_xpt('DEMO')
bmx   = load_xpt('BMX')
bpx   = load_xpt('BPX')
ghb   = load_xpt('GHB')
glu   = load_xpt('GLU')
tchol = load_xpt('TCHOL')
hdl   = load_xpt('HDL')
diq   = load_xpt('DIQ')
smq   = load_xpt('SMQ')
paq   = load_xpt('PAQ')

for name, df in [('DEMO',demo),('BMX',bmx),('BPX',bpx),('GHB',ghb),
                 ('GLU',glu),('TCHOL',tchol),('HDL',hdl),('DIQ',diq),
                 ('SMQ',smq),('PAQ',paq)]:
    print(f'{name:6}  {len(df):>6,} rows  {df.shape[1]:>3} cols')

## 2 · Merge on SEQN (participant ID)

In [ ]:
from functools import reduce

key_cols = {
    'demo':  ['SEQN','RIDAGEYR','RIAGENDR','DMDEDUC2','INDFMPIR','RIDRETH3'],
    'bmx':   ['SEQN','BMXBMI','BMXWT','BMXHT','BMXWAIST'],
    'bpx':   ['SEQN','BPXSY1','BPXDI1'],
    'ghb':   ['SEQN','LBXGH'],
    'glu':   ['SEQN','LBXGLU'],
    'tchol': ['SEQN','LBXTC'],
    'hdl':   ['SEQN','LBDHDD'],
    'diq':   ['SEQN','DIQ010'],
    'smq':   ['SEQN','SMQ020','SMQ040'],
    'paq':   ['SEQN','PAQ605','PAQ620','PAQ650','PAQ665','PAD680'],
}

dfs = [
    demo[key_cols['demo']], bmx[key_cols['bmx']], bpx[key_cols['bpx']],
    ghb[key_cols['ghb']], glu[key_cols['glu']], tchol[key_cols['tchol']],
    hdl[key_cols['hdl']], diq[key_cols['diq']], smq[key_cols['smq']],
    paq[key_cols['paq']],
]

merged = reduce(lambda a, b: a.merge(b, on='SEQN', how='left'), dfs)
merged = merged[merged['RIDAGEYR'] >= 18].copy()   # adults only
print(f'Merged dataset: {merged.shape[0]:,} adults  ×  {merged.shape[1]} columns')
merged.head(3)

## 3 · Missingness heatmap

In [ ]:
miss = merged.isnull().mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
miss[miss > 0].plot(kind='barh', color='steelblue', ax=ax)
ax.set_xlabel('Missing fraction')
ax.set_title('NHANES 2017-2018 — Missingness by variable')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

print('Variables with >50% missing:')
print(miss[miss > 0.5].to_string())

## 4 · Distribution of key continuous variables

In [ ]:
cont_vars = {
    'Age (yr)':           'RIDAGEYR',
    'BMI':                'BMXBMI',
    'Waist (cm)':         'BMXWAIST',
    'HbA1c (%)':          'LBXGH',
    'Fasting glucose':    'LBXGLU',
    'Total cholesterol':  'LBXTC',
    'HDL cholesterol':    'LBDHDD',
    'Systolic BP':        'BPXSY1',
    'Diastolic BP':       'BPXDI1',
}

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, (label, col) in zip(axes.flat, cont_vars.items()):
    data = merged[col].dropna()
    data = data[(data > data.quantile(0.01)) & (data < data.quantile(0.99))]
    ax.hist(data, bins=40, color='steelblue', edgecolor='none', alpha=0.85)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('')
    median = data.median()
    ax.axvline(median, color='tomato', ls='--', lw=1.2, label=f'Median={median:.1f}')
    ax.legend(fontsize=8)

fig.suptitle('Distribution of continuous variables — NHANES 2017-2018', y=1.01)
plt.tight_layout()
plt.show()

## 5 · Target variable analysis — Diabetes

In [ ]:
# DIQ010 == 1 → doctor-diagnosed diabetes
# Also flag if HbA1c >= 6.5 or fasting glucose >= 126
diab = merged.copy()
diab['has_diabetes'] = (
    (diab['DIQ010'] == 1) |
    (diab['LBXGH'] >= 6.5) |
    (diab['LBXGLU'] >= 126)
).astype(int)

prev = diab['has_diabetes'].mean()
print(f'Diabetes prevalence: {prev:.1%}  ({diab["has_diabetes"].sum():,} / {len(diab):,} adults)')

# By age group
diab['age_group'] = pd.cut(diab['RIDAGEYR'],
    bins=[18,35,45,55,65,120],
    labels=['18-35','36-45','46-55','56-65','65+'])

by_age = diab.groupby('age_group', observed=True)['has_diabetes'].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Prevalence overall
counts = diab['has_diabetes'].value_counts()
ax1.pie(counts, labels=['No diabetes','Diabetes'], autopct='%1.1f%%',
        colors=['#5b8dee','#ef4444'], startangle=90)
ax1.set_title('Overall diabetes prevalence')

# By age group
by_age.plot(kind='bar', color='steelblue', ax=ax2)
ax2.set_title('Diabetes prevalence by age group')
ax2.set_ylabel('Prevalence')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax2.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 6 · Correlation heatmap (continuous features)

In [ ]:
corr_cols = list(cont_vars.values()) + ['has_diabetes']
corr_labels = list(cont_vars.keys()) + ['Diabetes']

diab['has_diabetes_float'] = diab['has_diabetes'].astype(float)
plot_df = diab[corr_cols].copy()
plot_df.columns = corr_labels

corr = plot_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Pearson correlation — key NHANES variables')
plt.tight_layout()
plt.show()

## 7 · Key insights

From the exploration above:

- **Diabetes prevalence** ≈ 15-18% in the adult NHANES 2017-2018 sample.
- **HbA1c and fasting glucose** have the strongest positive correlation with diabetes.
- **Age** is a major risk factor — prevalence rises sharply after 45.
- **BMI and waist circumference** are highly correlated and both associated with diabetes risk.
- **HDL cholesterol** is negatively correlated with diabetes (protective).
- Lab values (HbA1c, glucose) have **~40-50% missingness** — handled by median imputation in the preprocessors.

➡️ Continue to `02_feature_engineering.ipynb`